# 04 — Deploy the React Dashboard

**Business Context:** HydraB needs a Vehicle 360 dashboard that combines fleet location, battery health, delivery pipeline, and CRM data in one place. Chris (Technical Champion) asked: "Is the app hosted within Snowflake?" — Yes, using Snowpark Container Services (SPCS).

**What this notebook does:**
- Deploys a pre-built React/Next.js dashboard as a container on Snowflake's compute pool
- The app reads directly from the GOLD schema — no data copies, no external APIs
- Exposes a public HTTPS endpoint for browser access

**Key features of the dashboard:**
- Fleet Map with GPS pins (answering: "Is this bus in its depot?")
- Sales Pipeline funnel (Opportunity stages + values)
- Vehicle telemetry detail pages (battery, speed, odometer)
- Embedded Cortex Agent chat (natural language queries)
- Depot weather conditions (from public API enrichment)

**Pain point addressed:** "We want to know if a bus is in a depot" (Jaco) — the Fleet Map shows real-time GPS positions overlaid with depot locations.

**Why SPCS?** The app runs inside Snowflake's security perimeter — no data leaves the platform, no external hosting to manage, and it uses Git-based source control (as Chris requested).


## Set Session Context
Resolves the per-user database name and sets the active warehouse.

In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;
USE WAREHOUSE HYDRAB_HOL_WH;

## Create SPCS Service
Deploys the pre-built React dashboard container from the shared image repository onto `HYDRAB_HOL_POOL`. Exposes port 3000 as a public HTTPS endpoint.

In [ ]:
-- Create the SPCS service from pre-built image
CREATE SERVICE IF NOT EXISTS DASHBOARD_SERVICE
  IN COMPUTE POOL HYDRAB_HOL_POOL
  FROM SPECIFICATION $$
  spec:
    containers:
    - name: dashboard
      image: /HYDRAB_HOL_SHARED/PUBLIC/IMAGE_REPO/hydrab-dashboard:v1
      env:
        SNOWFLAKE_DATABASE: HYDRAB_HOL_NHUVAERE
        HOSTNAME: "0.0.0.0"
      resources:
        requests:
          cpu: 0.5
          memory: 512M
        limits:
          cpu: 1
          memory: 1G
    endpoints:
    - name: dashboard
      port: 3000
      public: true
  $$
  MIN_INSTANCES = 1
  MAX_INSTANCES = 1;

## Check Service Status
Polls the service status. Re-run until it shows `RUNNING` (~1-2 minutes on first deploy).

In [ ]:
-- Wait for service to start (check status)
-- Run this cell a few times until status = RUNNING
SELECT SYSTEM$GET_SERVICE_STATUS('GOLD.DASHBOARD_SERVICE');

## Get Public URL
Shows the ingress URL for the dashboard. Open this in your browser.

In [ ]:
-- Get the public endpoint URL
SHOW ENDPOINTS IN SERVICE GOLD.DASHBOARD_SERVICE;

## Dashboard Deployed!

Copy the `ingress_url` from above and open it in your browser.
You should see the HydraB Vehicle 360 dashboard with:
- Overview stats (vehicles, pipeline, deliveries)
- Fleet Map with GPS pins
- Sales Pipeline funnel chart
- Vehicle telemetry detail pages

---

## What's Next: Extend with Cortex Code Desktop

The `react-app/` folder in this skill contains the full source code.
Open it in Cortex Code Desktop and ask CoCo to:
- Add a new page (e.g. Delivery Tracking)
- Connect charts to live Snowflake queries
- Improve the Fleet Map with vehicle popups
- Add the Cortex Agent chat to the dashboard

In [ ]:
-- Cleanup (run when done with the lab)
-- DROP SERVICE IF EXISTS GOLD.DASHBOARD_SERVICE;
-- DROP DATABASE IF EXISTS IDENTIFIER($user_db);